In [9]:
import pandas as pd
import numpy as np
from functions.eval import *
from scipy.stats import wilcoxon

In [10]:
model_pred_col = "Camelbert-MSA"
# model_pred_col = "AraBert"

In [11]:
lime_eval = pd.read_csv("data/xai_eval/xai_eval_lime_" + model_pred_col + ".csv")
lime_eval = lime_eval[['LIME_comprehensiveness', 'LIME_sufficiency', 'LIME_corr_loo', 'LIME_ins_AUC', 'LIME_del_AUC', 'LIME_combined']]

In [12]:
lime_eval

,LIME_comprehensiveness,LIME_sufficiency,LIME_corr_loo,LIME_ins_AUC,LIME_del_AUC,LIME_combined
0,0.455591,0.299552,0.453721,0.725979,0.414003,0.638975
1,0.340858,-0.013057,0.686264,0.809358,0.568659,0.687549
2,0.906829,0.899762,0.429294,0.444861,0.074873,0.618340
3,0.738834,-0.008222,0.352381,0.701583,0.391010,0.746764
4,0.589893,0.104099,0.104762,0.851822,0.686917,0.640616
...,...,...,...,...,...,...
3028,0.006474,-0.025030,0.308561,0.855552,0.501277,0.608012
3029,0.710399,0.751956,0.505394,0.455404,0.103254,0.612658
3030,0.727368,0.000142,0.072650,0.652369,0.548836,0.673417
3031,0.351587,0.421749,0.600000,0.675031,0.379315,0.605111


In [13]:
ensemble_eval = pd.read_csv("data/xai_eval/xai_eval_ensemble_" + model_pred_col + ".csv")
ensemble_eval = ensemble_eval[['EXAI_LIME_SHAP_mean_comprehensiveness',
       'EXAI_LIME_SHAP_mean_sufficiency', 'EXAI_LIME_SHAP_mean_corr_loo',
       'EXAI_LIME_SHAP_mean_ins_AUC', 'EXAI_LIME_SHAP_mean_del_AUC', 'EXAI_LIME_SHAP_mean_combined']]

In [14]:
ensemble_eval

,EXAI_LIME_SHAP_mean_comprehensiveness,EXAI_LIME_SHAP_mean_sufficiency,EXAI_LIME_SHAP_mean_corr_loo,EXAI_LIME_SHAP_mean_ins_AUC,EXAI_LIME_SHAP_mean_del_AUC,EXAI_LIME_SHAP_mean_combined
0,0.626092,0.739757,0.463768,0.696664,0.394514,0.584074
1,0.340858,-0.013057,0.582418,0.826140,0.538975,0.686458
2,0.891121,0.908650,0.364706,0.467691,0.094508,0.607601
3,0.910135,-0.003029,0.371429,0.722991,0.433201,0.777734
4,0.000544,0.253304,0.200000,0.846441,0.685665,0.501603
...,...,...,...,...,...,...
3028,0.006474,-0.025030,0.241654,0.847283,0.521978,0.595527
3029,0.764311,0.745716,0.464615,0.442480,0.090656,0.620546
3030,0.786679,-0.000105,0.114286,0.654672,0.628962,0.673928
3031,0.351588,0.421749,0.600000,0.675031,0.379315,0.605111


In [15]:
lime_eval["LIME_combined"].isna().sum(), ensemble_eval["EXAI_LIME_SHAP_mean_combined"].isna().sum()

(np.int64(1), np.int64(1))

In [18]:
# index of the rows with NaN values in the combined column
lime_eval[lime_eval["LIME_combined"].isna()].index

RangeIndex(start=303, stop=304, step=1)

In [20]:
ensemble_eval[ensemble_eval["EXAI_LIME_SHAP_mean_combined"].isna()].index

RangeIndex(start=889, stop=890, step=1)

In [23]:
eval_df = pd.concat([lime_eval, ensemble_eval], axis=1)

In [24]:
eval_df.dropna(inplace=True)

In [27]:
eval_df.isna().sum()

LIME_comprehensiveness                   0
LIME_sufficiency                         0
LIME_corr_loo                            0
LIME_ins_AUC                             0
LIME_del_AUC                             0
LIME_combined                            0
EXAI_LIME_SHAP_mean_comprehensiveness    0
EXAI_LIME_SHAP_mean_sufficiency          0
EXAI_LIME_SHAP_mean_corr_loo             0
EXAI_LIME_SHAP_mean_ins_AUC              0
EXAI_LIME_SHAP_mean_del_AUC              0
EXAI_LIME_SHAP_mean_combined             0
dtype: int64

In [28]:
eval_df.shape

(3031, 12)

In [29]:
effect = np.mean(eval_df["EXAI_LIME_SHAP_mean_combined"] - eval_df["LIME_combined"])
effect

np.float64(0.008442067222838592)

In [30]:
stat, p = wilcoxon(eval_df["LIME_combined"].values, eval_df["EXAI_LIME_SHAP_mean_combined"].values)
print(stat, p)

1869083.0 1.0766956238903175e-17


In [31]:
def bootstrap_ci(a, b, n=10000):
    diffs = []
    for _ in range(n):
        idx = np.random.choice(len(a), len(a), replace=True)
        diffs.append(np.mean(a[idx] - b[idx]))
    return np.percentile(diffs, [2.5, 97.5])

In [33]:
bootstrap_ci(eval_df["EXAI_LIME_SHAP_mean_combined"].values, eval_df["LIME_combined"].values)

array([0.00473644, 0.01210989])